# 🖼️ Lab 02 — Text-to-Image Generation with Qwen + OpenVINO

**Intel DevMeet Nagpur · Hands-On Workshop**

---

> **About this notebook**  
> This is a guided lab notebook. The instructor will walk through each section  
> live. Your job is to **read the explanations**, **study the hints**, and  
> **fill in the `TODO` blocks** before running the cell.

---

| Field | Details |
|---|---|
| 🗂️ Topic | Text-to-Image generation with an LLM prompt enhancer |
| 🛠️ Stack | Python · HuggingFace Transformers · OpenVINO · Diffusers |
| 🤖 LLM | [Qwen2-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2-0.5B-Instruct) |
| 🎨 Image Model | [Stable Diffusion v1-5](https://huggingface.co/runwayml/stable-diffusion-v1-5) |
| ⚙️ Accelerator | Intel OpenVINO Runtime |
| ⏱️ Estimated time | 90 – 120 min (includes ~15 min for initial model download + export) |


## 🎒 Prerequisites

Before starting this lab you should be comfortable with:

- **Python 3.10+** — basic syntax, functions, classes  
- **Jupyter notebooks** — running cells, understanding output  
- **PyTorch basics** — tensors, `torch.no_grad()`, `model.eval()`  
- **HuggingFace Transformers** — `AutoTokenizer`, `AutoModelForCausalLM`  
- *(Optional)* Familiarity with diffusion models helps but is not required

### ⚙️ Hardware recommendation

| Device | Minimum | Recommended |
|---|---|---|
| RAM | 8 GB | 16 GB |
| Storage | 10 GB free | 20 GB free |
| CPU | Any x86-64 | Intel Core i5+ / Xeon |
| GPU | — | Intel Arc / Intel iGPU (optional) |


## 🎯 Learning Objectives

By the end of this lab you will be able to:

1. **Explain** what OpenVINO is and why it accelerates inference on Intel hardware  
2. **Load and run** a Qwen language model using the `optimum-intel` OpenVINO backend  
3. **Write** a prompt-enhancement function that uses Qwen to turn a short phrase into a rich image description  
4. **Load** a Stable Diffusion pipeline exported to OpenVINO Intermediate Representation (IR)  
5. **Generate** images from text prompts entirely on CPU / Intel GPU using OpenVINO  
6. **Chain** the two models into a single end-to-end text-to-image pipeline  


## 🏗️ Pipeline Architecture

```
 User types a short prompt
         │
         ▼
 ┌──────────────────────────┐
 │    Qwen2 LLM             │  ← optimum-intel / OpenVINO backend
 │  (Prompt Enhancer)       │    Converts short prompt → detailed art description
 └──────────────────────────┘
         │  enhanced prompt
         ▼
 ┌──────────────────────────┐
 │  Stable Diffusion v1-5   │  ← diffusers / OVStableDiffusionPipeline
 │  (Text-to-Image)         │    Denoising diffusion process
 └──────────────────────────┘
         │
         ▼
      🖼️ Generated Image
```

### Why two models?
Stable Diffusion's text encoder (CLIP) is powerful but limited to ~77 tokens.  
Qwen can read a natural-language prompt and rewrite it into a much richer,  
vocabulary-optimised description that guides the diffusion model better.


---
## Part 1 — Environment Setup

### 1.1 Install required packages

We need the following **7** libraries:

| Package | Purpose |
|---|---|
| `openvino` | Core OpenVINO runtime |
| `optimum[openvino]` | HuggingFace ↔ OpenVINO bridge |
| `transformers` | Tokenizers and model configs |
| `diffusers` | Stable Diffusion pipeline |
| `accelerate` | HuggingFace model loading helpers |
| `Pillow` | Image display |
| `tqdm` | Progress bars |

📚 **Docs:**  
- OpenVINO install guide → https://docs.openvino.ai/2024/get-started/install-openvino.html  
- Optimum Intel → https://huggingface.co/docs/optimum/intel/openvino/overview  

---
✏️ **TODO 1.1** — Complete the `pip install` command below with the packages listed in the table above.  
💡 *Hint:* all packages are on PyPI; combine them in one `pip install` line separated by spaces.


In [ ]:
# ✏️ TODO 1.1 — Fill in the missing package names
# Replace each ??? with one package name from the table above (7 total)
# Note: some packages use extras in square brackets, e.g. optimum[openvino]
# Example: %pip install packageA "packageB[extra]" packageC

%pip install ??? "???" ??? ??? ??? ??? ???


In [ ]:
# ✅ REFERENCE (run this cell only if you are stuck or the instructor reveals it)
# %pip install openvino "optimum[openvino]" transformers diffusers accelerate Pillow tqdm


### 1.2 Import libraries

📚 **Docs:**  
- `openvino` Python API → https://docs.openvino.ai/2024/api/ie_python_api/api.html  
- `OVModelForCausalLM` → https://huggingface.co/docs/optimum/intel/openvino/reference#optimum.intel.OVModelForCausalLM  
- `OVStableDiffusionPipeline` → https://huggingface.co/docs/optimum/intel/openvino/reference#optimum.intel.OVStableDiffusionPipeline  

---
✏️ **TODO 1.2** — Fill in the missing import targets marked with `???`.  
💡 *Hints:*  
- Import the OpenVINO `Core` class from the `openvino` package (just write `Core`, not `openvino.Core`)  
- Import `AutoTokenizer` from `transformers`  
- Import `OVModelForCausalLM` from `optimum.intel`  
- Import `OVStableDiffusionPipeline` from `optimum.intel`  


In [ ]:
# Standard library
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from typing import Optional

# OpenVINO core
from openvino import ???          # ✏️ TODO: import the Core class

# HuggingFace Transformers
from transformers import ???      # ✏️ TODO: import AutoTokenizer

# Optimum-Intel (OpenVINO backend for HuggingFace models)
from optimum.intel import ???, ??? # ✏️ TODO: import OVModelForCausalLM AND OVStableDiffusionPipeline

# Utilities
from PIL import Image
import numpy as np
from tqdm.auto import tqdm
import torch
import time


In [ ]:
# ✅ REFERENCE
# from openvino import Core
# from transformers import AutoTokenizer
# from optimum.intel import OVModelForCausalLM, OVStableDiffusionPipeline


### 1.3 Verify installations

Run the cell below (no changes needed) to confirm all packages are installed correctly.


In [ ]:
import openvino
import transformers
import diffusers
import optimum
import PIL

print(f"OpenVINO version   : {openvino.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Diffusers version  : {diffusers.__version__}")
print(f"Optimum version    : {optimum.__version__}")
print(f"Pillow version     : {PIL.__version__}")
print("\n✅ All packages loaded successfully!")


---
## Part 2 — OpenVINO Fundamentals

### What is OpenVINO?

**OpenVINO™** (Open Visual Inference and Neural network Optimization) is Intel's  
open-source toolkit for optimising and deploying deep-learning models.

Key ideas:
1. **Model conversion** — convert PyTorch / TensorFlow / ONNX models to a hardware-neutral IR (`.xml` + `.bin`)  
2. **Plugin architecture** — a single API runs on CPU, iGPU, discrete GPU, VPU, FPGA  
3. **INT8 / FP16 quantisation** — automatic precision reduction for 2-4× speed-up with minimal accuracy loss  

```
PyTorch model  ──► ov.convert_model() ──► IR (.xml + .bin) ──► ov.compile_model() ──► Inference
```

📚 **Docs:**  
- OpenVINO overview → https://docs.openvino.ai/2024/about-openvino/overview.html  
- `openvino.Core` API → https://docs.openvino.ai/2024/api/ie_python_api/_autosummary/openvino.Core.html  


### 2.1 Discover available hardware

The `openvino.Core` object is the entry point for the runtime.  
`core.available_devices` returns a list of device names you can run on.

| Device string | Meaning |
|---|---|
| `CPU` | Intel (or any x86) CPU |
| `GPU` | Intel integrated / discrete GPU |
| `GPU.0`, `GPU.1` | Multiple Intel GPUs |
| `AUTO` | OpenVINO picks the fastest available device |

📚 **Docs:** https://docs.openvino.ai/2024/openvino-workflow/running-inference/inference-devices-and-modes.html  

---
✏️ **TODO 2.1** — Create a `Core` object and print the list of available devices.  
💡 *Hints:*  
- Instantiate `Core()` with no arguments  
- Access the `.available_devices` attribute  
- Loop over the list and print each device with its full name using `core.get_property(device, "FULL_DEVICE_NAME")`  


In [ ]:
# ✏️ TODO 2.1 — Instantiate Core and list devices

core = ???()          # Create an OpenVINO Core instance

print("Available inference devices:")
for device in core.???:   # ✏️ access the available_devices attribute
    try:
        full_name = core.get_property(device, "FULL_DEVICE_NAME")
    except Exception:
        full_name = device
    print(f"  • {device:10s}  →  {full_name}")


In [ ]:
# ✅ REFERENCE
# core = Core()
# for device in core.available_devices:
#     ...


### 2.2 Choose your inference device

For this lab we default to `"CPU"` which works on every machine.  
If you see `GPU` in the list above, change the variable to `"GPU"` for faster inference.

📚 Device selection guide → https://docs.openvino.ai/2024/openvino-workflow/running-inference/inference-devices-and-modes/auto-device-selection.html


In [ ]:
# Change this to "GPU" or "AUTO" if your machine has an Intel GPU
DEVICE = "CPU"
print(f"Selected inference device: {DEVICE}")


---
## Part 3 — Load the Qwen2 Language Model with OpenVINO

### What is Qwen2?

**Qwen2** is a family of large language models developed by Alibaba Group.  
The smallest variant, **Qwen2-0.5B-Instruct**, has only 500 million parameters —  
small enough to run comfortably on a laptop CPU.

We use the `optimum-intel` library to load Qwen2 **directly with an OpenVINO backend**  
using `OVModelForCausalLM`. This is equivalent to `AutoModelForCausalLM` but the  
weights are automatically exported to OpenVINO IR format on first load.

```python
# Normal HuggingFace loading (PyTorch)
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B-Instruct")

# OpenVINO-accelerated loading (OV runtime, no GPU memory needed)
from optimum.intel import OVModelForCausalLM
model = OVModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B-Instruct", export=True)
```

�� **Docs:**  
- Qwen2 model card → https://huggingface.co/Qwen/Qwen2-0.5B-Instruct  
- OVModelForCausalLM → https://huggingface.co/docs/optimum/intel/openvino/reference#optimum.intel.OVModelForCausalLM  


### 3.1 Configuration

Set the model name and local cache directory once so all later cells use the same values.


In [ ]:
# Model name on HuggingFace Hub
QWEN_MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"

# Directory to save the exported OpenVINO model (avoids re-exporting on every run)
QWEN_OV_DIR = Path("models/qwen2-0.5b-instruct-ov")

print(f"Qwen2 model : {QWEN_MODEL_ID}")
print(f"OV save dir : {QWEN_OV_DIR}")


### 3.2 Load the tokenizer

The **tokenizer** converts raw text into token IDs that the model can process.  
Qwen2 uses a Byte-Pair Encoding (BPE) tokenizer with a 151,643-token vocabulary.

📚 **Docs:**  
- `AutoTokenizer.from_pretrained` → https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.AutoTokenizer.from_pretrained  

---
✏️ **TODO 3.2** — Load the tokenizer for `QWEN_MODEL_ID`.  
💡 *Hints:*  
- Use `AutoTokenizer.from_pretrained(model_name)`  
- Pass `trust_remote_code=True` (required for Qwen models)  


In [ ]:
# ✏️ TODO 3.2 — Load the Qwen2 tokenizer

tokenizer = ???.from_pretrained(
    ???,                    # ✏️ the model id variable
    trust_remote_code=???   # ✏️ True or False?
)

print(f"Tokenizer class    : {tokenizer.__class__.__name__}")
print(f"Vocabulary size    : {tokenizer.vocab_size:,}")
print(f"Model max length   : {tokenizer.model_max_length}")


In [ ]:
# ✅ REFERENCE
# tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)


### 3.3 Load the Qwen2 model with OpenVINO

`OVModelForCausalLM.from_pretrained` with `export=True` will:
1. Download the original PyTorch weights from HuggingFace Hub  
2. Convert them to OpenVINO IR format (`.xml` + `.bin`)  
3. Save the IR to the specified directory  
4. Compile the model for the selected device  

> ⚠️ **First run**: export takes **2–5 minutes** and requires ~4 GB disk space.  
> **Subsequent runs**: model loads directly from the saved IR in seconds.

📚 **Docs:**  
- Exporting a model → https://huggingface.co/docs/optimum/intel/openvino/export  

---
✏️ **TODO 3.3** — Load (or export) the Qwen2 model with OpenVINO.  
💡 *Hints:*  
- If `QWEN_OV_DIR` already exists, load from disk with `export=False`  
- Otherwise load from HuggingFace with `export=True` and pass `save_directory=QWEN_OV_DIR`  
- Always pass `device=DEVICE` (use the variable you set earlier)  
- Pass `trust_remote_code=True`  


In [ ]:
# ✏️ TODO 3.3 — Load the Qwen2 model

if QWEN_OV_DIR.exists():
    # Model was already exported — load it directly from disk (fast)
    print(f"Loading pre-exported OpenVINO model from {QWEN_OV_DIR} ...")
    qwen_model = OVModelForCausalLM.from_pretrained(
        ???,           # ✏️ path to saved OV model
        device=???,    # ✏️ use DEVICE variable
        trust_remote_code=True
    )
else:
    # First run: export from HuggingFace to OpenVINO IR
    print(f"Exporting {QWEN_MODEL_ID} to OpenVINO IR — this takes a few minutes ...")
    qwen_model = OVModelForCausalLM.from_pretrained(
        ???,                        # ✏️ HuggingFace model id
        export=???,                 # ✏️ True to trigger export
        device=???,                 # ✏️ use DEVICE variable
        trust_remote_code=True,
        save_directory=???          # ✏️ where to save the IR
    )

print("\n✅ Qwen2 model loaded on:", qwen_model.device)


In [ ]:
# ✅ REFERENCE
# if QWEN_OV_DIR.exists():
#     qwen_model = OVModelForCausalLM.from_pretrained(
#         QWEN_OV_DIR, device=DEVICE, trust_remote_code=True)
# else:
#     qwen_model = OVModelForCausalLM.from_pretrained(
#         QWEN_MODEL_ID, export=True, device=DEVICE,
#         trust_remote_code=True, save_directory=QWEN_OV_DIR)


### 3.4 Quick sanity check — run a simple text generation

Before building the full pipeline, verify the model works by generating a short response.

📚 **Docs:**  
- `model.generate()` → https://huggingface.co/docs/transformers/main_classes/text_generation  
- Qwen2 chat template → https://huggingface.co/Qwen/Qwen2-0.5B-Instruct#quickstart  


In [ ]:
def qwen_generate(prompt: str, max_new_tokens: int = 200) -> str:
    """Run a single inference pass through Qwen2 and return the generated text."""
    # Build the chat-format message list expected by Qwen2 Instruct
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": prompt}
    ]
    # Apply the chat template to get raw token IDs
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt")

    with torch.no_grad():
        generated_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    # Decode only the newly generated tokens (strip the input prompt)
    new_tokens = generated_ids[:, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens[0], skip_special_tokens=True)

# Quick test
print("Testing Qwen2 inference ...")
response = qwen_generate("What is the capital of India?", max_new_tokens=50)
print(f"Response: {response}")


---
## Part 4 — Prompt Engineering with Qwen2

### Why does prompt quality matter for image generation?

Stable Diffusion interprets prompts literally through a CLIP text encoder.  
A short prompt like *"a dog"* is valid, but the model gets much more guidance from  
a richer prompt like:  

> *"a golden retriever sitting on a grassy hill, soft morning light, cinematic,  
> highly detailed, 8k, artstation quality"*

Qwen2 can do this rewriting for us automatically.

### Prompt engineering vocabulary

Common terms that improve Stable Diffusion output:

| Category | Example terms |
|---|---|
| **Style** | *oil painting, digital art, watercolor, photorealistic, sketch* |
| **Quality** | *masterpiece, highly detailed, 8k, sharp focus, studio quality* |
| **Lighting** | *golden hour, soft light, dramatic lighting, neon lights* |
| **Composition** | *wide angle, portrait, close-up, rule of thirds* |
| **Artists** | *by Greg Rutkowski, by Alphonse Mucha* |
| **Negative concepts** | blurry, low quality, distorted *(used in negative prompt)* |

📚 **Docs:**  
- Stable Diffusion prompt guide → https://stable-diffusion-art.com/prompt-guide/  
- Qwen2 generation parameters → https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.GenerationConfig  


### 4.1 Write the prompt enhancement function

✏️ **TODO 4.1** — Complete the function below that uses Qwen2 to turn a short user  
input into a detailed Stable Diffusion prompt.

💡 *Hints:*  
- Write a clear system message that instructs Qwen to act as a prompt engineer  
- The system message should tell the model to output **only** the enhanced prompt (no explanation)  
- Limit output to ~150 new tokens — image prompts should not be too long  
- Use the `qwen_generate` helper you already wrote, or call the model directly  


In [ ]:
def enhance_prompt(user_input: str) -> str:
    """
    Use Qwen2 to rewrite a short text description into a rich,
    detailed Stable Diffusion prompt.

    Parameters
    ----------
    user_input : str
        Short description, e.g. "a cat in space"

    Returns
    -------
    str
        Enhanced prompt with style, quality, and composition details.

    Note
    ----
    Because ``do_sample=True`` is used, the enhanced prompt may differ
    slightly on each call. Pass a fixed ``temperature`` or use
    ``do_sample=False`` for fully reproducible output.
    """
    # ✏️ TODO 4.1a — Write a system message that instructs Qwen2 to act as
    # a Stable Diffusion prompt engineer.
    # Tell it to return ONLY the enhanced prompt with no explanations.
    system_msg = """???"""

    # ✏️ TODO 4.1b — Write the user message that includes the original input
    user_msg = f"???{user_input}???"

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_msg}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt")

    with torch.no_grad():
        out = qwen_model.generate(
            **inputs,
            max_new_tokens=???,    # ✏️ TODO 4.1c — pick a sensible token limit
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    enhanced = tokenizer.decode(new_tokens[0], skip_special_tokens=True).strip()
    return enhanced


In [ ]:
# ✅ REFERENCE — only look if you are truly stuck!
# system_msg = (
#     "You are an expert Stable Diffusion prompt engineer. "
#     "When the user gives you a short description, rewrite it into a "
#     "detailed, high-quality image generation prompt. Include style, "
#     "lighting, composition, and quality keywords. "
#     "Output ONLY the enhanced prompt — no explanations, no extra text."
# )
# user_msg = f"Enhance this image prompt: {user_input}"
# max_new_tokens = 150


### 4.2 Test the prompt enhancer

Run the cell below and observe how Qwen2 transforms simple inputs.  
Try changing `test_prompts` to your own ideas!


In [ ]:
test_prompts = [
    "a cat sitting in space",
    "ancient temple in a jungle",
    "futuristic city at night",
]

for short_prompt in test_prompts:
    print(f"\n{'─'*60}")
    print(f"  Original : {short_prompt}")
    enhanced = enhance_prompt(short_prompt)
    print(f"  Enhanced : {enhanced}")


---
## Part 5 — Text-to-Image Generation with Stable Diffusion + OpenVINO

### How does Stable Diffusion work?

Stable Diffusion is a **latent diffusion model** (LDM) that generates images by  
progressively denoising a random noise tensor.

```
Random Gaussian noise  →  [T denoising steps]  →  Latent image  →  VAE decode  →  Pixel image
                                  ↑
                     Text prompt (via CLIP encoder)
```

**Key components:**
- **CLIP Text Encoder** — converts text tokens → embeddings  
- **U-Net** — the denoising backbone; predicts noise at each step  
- **VAE** (Variational Autoencoder) — compresses / decompresses images  
- **Scheduler** — controls the denoising step sequence (DDIM, PNDM, DPM++, …)

📚 **Docs:**  
- Stable Diffusion explained → https://huggingface.co/blog/stable_diffusion  
- `OVStableDiffusionPipeline` → https://huggingface.co/docs/optimum/intel/openvino/reference#optimum.intel.OVStableDiffusionPipeline  
- Schedulers → https://huggingface.co/docs/diffusers/api/schedulers/overview  


### 5.1 Load the Stable Diffusion pipeline with OpenVINO

`OVStableDiffusionPipeline` works exactly like HuggingFace `StableDiffusionPipeline`  
but all four sub-models (text encoder, U-Net, VAE encoder, VAE decoder) are  
compiled to OpenVINO IR and run via the OpenVINO runtime.

> ⚠️ **First run**: export + compilation takes **5–10 minutes** and ~6 GB disk space.  
> **Subsequent runs** load in ~30 seconds.

📚 **Docs:**  
- OVStableDiffusionPipeline → https://huggingface.co/docs/optimum/intel/openvino/reference#optimum.intel.OVStableDiffusionPipeline  

---
✏️ **TODO 5.1** — Load the Stable Diffusion pipeline similarly to how you loaded Qwen.  
💡 *Hints:*  
- Model ID: `"runwayml/stable-diffusion-v1-5"`  
- Save directory: `Path("models/sd-v1-5-ov")`  
- Use `OVStableDiffusionPipeline.from_pretrained(...)` with `export=True` on first run  
- Pass `device=DEVICE`  


In [ ]:
SD_MODEL_ID  = "runwayml/stable-diffusion-v1-5"
SD_OV_DIR    = Path("models/sd-v1-5-ov")

# ✏️ TODO 5.1 — Load the Stable Diffusion pipeline with OpenVINO
# Mirror the pattern you used in Part 3 for Qwen.

if ???.exists():  # ✏️ check if OV model already saved
    print(f"Loading pre-exported Stable Diffusion from {SD_OV_DIR} ...")
    sd_pipe = OVStableDiffusionPipeline.from_pretrained(
        ???,         # ✏️ path
        device=???   # ✏️ device
    )
else:
    print(f"Exporting {SD_MODEL_ID} to OpenVINO IR — grab a coffee, this takes ~5 minutes ...")
    sd_pipe = OVStableDiffusionPipeline.from_pretrained(
        ???,                    # ✏️ HuggingFace model id
        export=???,             # ✏️ True
        device=???,             # ✏️ DEVICE
        save_directory=???      # ✏️ SD_OV_DIR
    )

print("\n✅ Stable Diffusion pipeline ready!")


In [ ]:
# ✅ REFERENCE
# if SD_OV_DIR.exists():
#     sd_pipe = OVStableDiffusionPipeline.from_pretrained(SD_OV_DIR, device=DEVICE)
# else:
#     sd_pipe = OVStableDiffusionPipeline.from_pretrained(
#         SD_MODEL_ID, export=True, device=DEVICE, save_directory=SD_OV_DIR)


### 5.2 Understand the generation parameters

Before generating images, let's understand the key knobs:

| Parameter | Type | Typical range | Effect |
|---|---|---|---|
| `prompt` | `str` | — | Positive guidance: what to include |
| `negative_prompt` | `str` | — | What to exclude (blurry, distorted, …) |
| `num_inference_steps` | `int` | 20 – 50 | More steps → better quality, slower |
| `guidance_scale` | `float` | 5 – 15 | Higher = stronger prompt adherence |
| `height` / `width` | `int` | 512 (multiple of 8) | Output image resolution |
| `generator` | `torch.Generator` | — | Random seed for reproducibility |

📚 **Docs:**  
- Pipeline call signature → https://huggingface.co/docs/diffusers/api/pipelines/stable_diffusion/text2img#diffusers.StableDiffusionPipeline.__call__  


In [ ]:
# Default generation settings — adjust these as you experiment

DEFAULT_NEGATIVE_PROMPT = (
    "blurry, low quality, distorted, deformed, ugly, bad anatomy, "
    "watermark, text, signature, out of frame"
)

DEFAULT_STEPS    = 25     # inference steps (20-30 is a good balance for demos)
DEFAULT_GUIDANCE = 7.5    # guidance scale
IMAGE_SIZE       = 512    # height and width (must be multiples of 8)
SEED             = 42     # fixed seed for reproducibility

print("Generation defaults set:")
print(f"  Steps     : {DEFAULT_STEPS}")
print(f"  Guidance  : {DEFAULT_GUIDANCE}")
print(f"  Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"  Seed      : {SEED}")


### 5.3 Generate your first image

✏️ **TODO 5.3** — Complete the `generate_image` function to call the Stable Diffusion pipeline.

💡 *Hints:*  
- Call `sd_pipe(prompt=..., negative_prompt=..., ...)` — it returns a `StableDiffusionPipelineOutput`  
- Access the generated image with `.images[0]`  
- Pass `generator=torch.Generator().manual_seed(seed)` for reproducibility  


In [ ]:
def generate_image(
    prompt: str,
    negative_prompt: str = DEFAULT_NEGATIVE_PROMPT,
    num_inference_steps: int = DEFAULT_STEPS,
    guidance_scale: float = DEFAULT_GUIDANCE,
    height: int = IMAGE_SIZE,
    width: int = IMAGE_SIZE,
    seed: int = SEED,
) -> Image.Image:
    """
    Generate an image from a text prompt using Stable Diffusion + OpenVINO.

    Returns a PIL.Image.Image object.
    """
    generator = torch.Generator().manual_seed(seed)

    print(f"Generating image with {num_inference_steps} steps on {DEVICE} ...")
    t0 = time.time()

    # ✏️ TODO 5.3 — Call the Stable Diffusion pipeline
    result = sd_pipe(
        prompt=???,                          # ✏️ positive prompt
        negative_prompt=???,                 # ✏️ negative prompt
        num_inference_steps=???,             # ✏️ number of denoising steps
        guidance_scale=???,                  # ✏️ CFG scale
        height=???,                          # ✏️ image height
        width=???,                           # ✏️ image width
        generator=???                        # ✏️ torch generator for reproducibility
    )

    elapsed = time.time() - t0
    print(f"✅ Done in {elapsed:.1f}s")

    # ✏️ TODO 5.3b — Extract the image from the result
    return result.???[0]


In [ ]:
# ✅ REFERENCE
# result = sd_pipe(
#     prompt=prompt,
#     negative_prompt=negative_prompt,
#     num_inference_steps=num_inference_steps,
#     guidance_scale=guidance_scale,
#     height=height,
#     width=width,
#     generator=generator
# )
# return result.images[0]


In [ ]:
# Quick test — generate with a simple prompt (no Qwen enhancement yet)
simple_prompt = "a red sports car on a mountain road, photorealistic, 8k"
test_image = generate_image(simple_prompt)

# Display inline in the notebook
display(test_image)
test_image.save("output_simple.png")
print("Image saved to output_simple.png")


---
## Part 6 — End-to-End Pipeline

Now we chain everything together:  
**User prompt → Qwen2 enhancement → Stable Diffusion → Image**

📚 **Related reading:**  
- Prompt-to-Prompt paper → https://arxiv.org/abs/2208.01626  
- Creative prompt strategies → https://huggingface.co/blog/stable_diffusion#tips-and-tricks  


### 6.1 Build the complete pipeline function

✏️ **TODO 6.1** — Combine `enhance_prompt` and `generate_image` into a single  
`text_to_image` function.

💡 *Hints:*  
- Call `enhance_prompt(user_input)` first to get the detailed prompt  
- Print both the original and enhanced prompts (useful for debugging)  
- Then call `generate_image(enhanced_prompt)` with any extra kwargs  
- Return the PIL image  


In [ ]:
def text_to_image(
    user_input: str,
    use_enhancement: bool = True,
    **generate_kwargs
) -> Image.Image:
    """
    Full pipeline: short text → enhanced prompt → generated image.

    Parameters
    ----------
    user_input : str
        Short, natural-language description.
    use_enhancement : bool
        If True, use Qwen2 to enhance the prompt first.
    **generate_kwargs
        Extra keyword arguments forwarded to generate_image().

    Returns
    -------
    PIL.Image.Image
    """
    print(f"\n{'═'*60}")
    print(f"  Input  : {user_input}")

    if use_enhancement:
        # ✏️ TODO 6.1a — call enhance_prompt
        prompt = ???(user_input)
        print(f"  Enhanced: {prompt}")
    else:
        prompt = user_input
        print(f"  Prompt : {prompt}  (no enhancement)")

    print(f"{'─'*60}")

    # ✏️ TODO 6.1b — call generate_image with the (enhanced) prompt
    image = ???(prompt, **generate_kwargs)

    return image


In [ ]:
# ✅ REFERENCE
# prompt = enhance_prompt(user_input)
# image  = generate_image(prompt, **generate_kwargs)


### 6.2 Try it out!

Run the pipeline with different inputs. Compare enhanced vs non-enhanced prompts.


In [ ]:
# ── Experiment 1: With enhancement ──────────────────────────────────────────
input1 = "a lighthouse during a storm"
img1_enhanced = text_to_image(input1, use_enhancement=True,  seed=100)
img1_plain    = text_to_image(input1, use_enhancement=False, seed=100)

# Side-by-side display
import PIL.ImageDraw, PIL.ImageFont
def side_by_side(img_a, label_a, img_b, label_b, img_size=512):
    canvas = Image.new("RGB", (img_size * 2 + 20, img_size + 40), "white")
    canvas.paste(img_a.resize((img_size, img_size)), (0, 40))
    canvas.paste(img_b.resize((img_size, img_size)), (img_size + 20, 40))
    draw = PIL.ImageDraw.Draw(canvas)
    draw.text((10, 10),            label_a, fill="black")
    draw.text((img_size + 30, 10), label_b, fill="black")
    return canvas

comparison = side_by_side(img1_plain, "No enhancement", img1_enhanced, "With Qwen2 enhancement")
display(comparison)
comparison.save("comparison.png")
print("Side-by-side saved to comparison.png")


In [ ]:
# ── Experiment 2: Your own prompt ───────────────────────────────────────────
# ✏️ Change the prompt below to anything you like and run the full pipeline!

my_prompt = "a dragon flying over a futuristic city"   # ← change me!
my_image   = text_to_image(my_prompt, use_enhancement=True, num_inference_steps=30, seed=7)
display(my_image)
my_image.save("my_output.png")


---
## Part 7 — Experiments & Challenges 🧪

Now it's your turn to explore! Pick any of the challenges below.

---

### 🟢 Challenge A — Guidance Scale Sweep (Easy)

Generate the **same prompt** at different `guidance_scale` values (e.g. 3, 7, 12, 15)  
and display them in a grid to see how it affects adherence to the prompt.

💡 *Hint:* Use a list comprehension to generate multiple images, then use `PIL.Image.new`  
to stitch them into a grid.


In [ ]:
# ✏️ Challenge A — Guidance scale sweep
prompt_A = enhance_prompt("an astronaut on the moon playing guitar")

guidance_values = [3.0, 7.5, 12.0, 15.0]
images_A = []

for gs in guidance_values:
    print(f"Generating with guidance_scale={gs} ...")
    # ✏️ TODO: Call generate_image with the current guidance_scale value
    img = generate_image(???, guidance_scale=???, num_inference_steps=20, seed=42)
    images_A.append((gs, img))

# Display grid
grid_w, grid_h = 512 * 2, 512 * 2
grid = Image.new("RGB", (grid_w, grid_h), "white")
for idx, (gs, img) in enumerate(images_A):
    r, c = divmod(idx, 2)
    grid.paste(img.resize((512, 512)), (c * 512, r * 512))

display(grid)
grid.save("challenge_a_guidance_sweep.png")


---

### 🟡 Challenge B — Scheduler Comparison (Medium)

Stable Diffusion supports multiple schedulers that affect output quality and speed.  
Swap the scheduler and observe the difference.

Common schedulers in `diffusers`:
- `DDIMScheduler` — deterministic, good quality  
- `PNDMScheduler` — default for SD v1.5  
- `DPMSolverMultistepScheduler` — high quality in fewer steps  
- `EulerDiscreteScheduler` — fast and sharp  

📚 **Docs:**  
- Scheduler overview → https://huggingface.co/docs/diffusers/api/schedulers/overview  


In [ ]:
# ✏️ Challenge B — Try a different scheduler
from diffusers import DDIMScheduler, DPMSolverMultistepScheduler

prompt_B = enhance_prompt("a samurai in cherry blossom forest at sunset")

# ── Step 1: Generate with current (default) scheduler
img_default = generate_image(prompt_B, num_inference_steps=25, seed=42)

# ── Step 2: Swap the scheduler
# ✏️ TODO: Replace the scheduler on sd_pipe with DPMSolverMultistepScheduler
# Hint: sd_pipe.scheduler = DPMSolverMultistepScheduler.from_config(sd_pipe.scheduler.config)
sd_pipe.scheduler = ???.from_config(  # ✏️ which scheduler class?
    sd_pipe.scheduler.config          # .config holds the current scheduler settings
)

# ── Step 3: Generate again (DPM++ only needs ~15 steps for similar quality)
img_dpm = generate_image(prompt_B, num_inference_steps=15, seed=42)

comparison_B = side_by_side(img_default, "Default (25 steps)", img_dpm, "DPM++ (15 steps)")
display(comparison_B)


---

### 🔴 Challenge C — Batch Generation with a Story (Hard)

Write a short story (3–5 sentences) and generate one image per sentence to create  
a visual storyboard.

💡 *Hints:*  
- Split the story into sentences using Python's `str.split('.')`  
- Run the full `text_to_image` pipeline for each sentence  
- **You must implement the storyboard assembly yourself** — calculate the grid layout,  
  paste each image at the correct `(x, y)` offset using `PIL.Image.paste()`,  
  and label each scene with `PIL.ImageDraw.Draw.text()`  
- Save the final contact sheet as `storyboard.png`  


In [ ]:
# ✏️ Challenge C — Visual storyboard

story = """
A lone astronaut lands on an alien planet with two suns.
She discovers a forest of glowing crystal trees.
In the distance, an ancient alien city emerges from the fog.
Inside the city, she finds a message left by a previous explorer.
She sends a beacon home, her mission complete.
""".strip()

sentences = [s.strip() for s in story.split('.') if s.strip()]
storyboard_images = []

for i, sentence in enumerate(sentences, 1):
    print(f"\nScene {i}/{len(sentences)}: {sentence}")
    # ✏️ TODO C-a: Call the full pipeline for each sentence
    img = ???(sentence, use_enhancement=True, num_inference_steps=20, seed=i*10)
    storyboard_images.append(img)

# ✏️ TODO C-b: Assemble a 2-column grid storyboard and display it
# You need to:
#   1. Decide the grid dimensions (n_cols=2, n_rows = ceil(len / n_cols))
#   2. Create a blank PIL Image large enough for the grid
#   3. Paste each image at position (col * img_w, row * img_h)
#   4. Add a scene number label with PIL.ImageDraw
#   5. Save to "storyboard.png" and call display()
import math
n_cols = 2
n_rows = ???                    # ✏️ use math.ceil(len(storyboard_images) / n_cols)
img_w, img_h = ???, ???        # ✏️ each thumbnail should be 512x512
canvas = Image.new(???, (???, ???), "white")  # ✏️ mode, width, height
draw   = PIL.ImageDraw.Draw(canvas)
for idx, img in enumerate(storyboard_images):
    row, col = divmod(idx, n_cols)
    x, y = ???, ???            # ✏️ compute pixel offset from row, col, img_w, img_h
    canvas.paste(img.resize((img_w, img_h)), (x, y))
    draw.text((x + 10, y + 10), f"Scene {idx+1}", fill="white")
display(canvas)
canvas.save("storyboard.png")
print("Storyboard saved to storyboard.png")


---
## Part 8 — Summary & Next Steps

### ✅ What you accomplished today

| Step | You learned |
|---|---|
| Environment setup | Installing and importing OpenVINO + HuggingFace packages |
| OpenVINO Core | Discovering devices, understanding the IR format |
| Qwen2 with OpenVINO | Loading / exporting a causal LLM using `optimum-intel` |
| Prompt engineering | Using an LLM to expand short descriptions into rich prompts |
| Stable Diffusion + OV | Loading and running a diffusion pipeline entirely on Intel hardware |
| End-to-end pipeline | Chaining LLM → Diffusion Model for text-to-image generation |

---

### 🔭 Where to go next

| Topic | Resource |
|---|---|
| OpenVINO model zoo | https://github.com/openvinotoolkit/open_model_zoo |
| OpenVINO notebooks | https://github.com/openvinotoolkit/openvino_notebooks |
| SDXL + OpenVINO | https://huggingface.co/docs/optimum/intel/openvino/stable-diffusion-xl |
| Qwen2-VL (Vision-Language) | https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct |
| INT8 quantisation with NNCF | https://docs.openvino.ai/2024/openvino-workflow/model-optimization.html |
| ControlNet + OpenVINO | https://huggingface.co/lllyasviel/ControlNet |
| IP-Adapter | https://ip-adapter.github.io/ |
| Gradio UI for your pipeline | https://www.gradio.app/guides/quickstart |

---

### 💡 Key takeaways

1. **OpenVINO bridges** HuggingFace models to Intel hardware with minimal code changes.  
2. **`optimum-intel`** makes exporting any HuggingFace model to OV IR a one-liner.  
3. **Prompt quality** strongly influences image quality — LLM-based enhancement is a practical technique.  
4. **Modular pipelines** (LLM + diffusion model) are more flexible than monolithic solutions.  
5. Running inference **locally on CPU** is practical for small models thanks to OpenVINO optimisations.

---

*Thank you for attending Intel DevMeet Nagpur! 🎉*  
*Questions? Reach out on the Intel DevMesh community → https://devmesh.intel.com*
